# 自己寫 LoRA fine-tune

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir("/content/drive/MyDrive/++DL/04_project_face"))

Mounted at /content/drive
['02_LoRA_training.ipynb', 'training_data', '01_prepare_dataset.ipynb', 'lora_output_test', 'loss_curve.png', 'lora_output_test_p1', 'loss_comparison.png', 'evaluation_grid_p1.png', 'comparison_4col.png', 'evaluation_grid.png', '03_p1_LoRA_inference_evaluate.ipynb', '03_LoRA_inference_evaluate.ipynb', 'eval_6metrics', '03_face_LoRA_inference_evaluate.ipynb', '01_prepare_dataset_morph.ipynb', 'training_data_morph', 'lora_output_morph_test_p2', 'eval_6metrics_morph', '03_face_morph_LoRA_inference_evaluate.ipynb']


In [2]:
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.5 MB/s eta 0:00:00


可以修改的參數: LR, warmup, grad_accum, max_steps, lora_rank.

og: 1e-4, 50, 4, 500, 4

p1: 5e-5, 100, 2, 500, 4

p2: 5e-5, 10, 2, 100, 4

In [3]:
# ================================================
# 把這整段存成一個 cell 跑，或存成 train_lora.py
# ================================================
!pip install -q peft

# ---- 寫檔案 ----
train_script = '''
import os, json, torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler, AutoencoderKL, DDPMScheduler
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

# ============ Config ============
DATA_DIR       = "/content/drive/MyDrive/++DL/04_project_face/training_data_morph"
OUTPUT_DIR     = "/content/drive/MyDrive/++DL/04_project_face/lora_output_morph_test_p1"
LORA_RANK      = 4
LR             = 5e-5 # 5e-5
MAX_STEPS      = 500        # 測試用 100，正式用 500
BATCH_SIZE     = 1
GRAD_ACCUM     = 2  # 2
RESOLUTION     = 512
MIXED_PREC     = True
SEED           = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
device = "cuda"

# ============ Dataset ============
class SketchPhotoDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        with open(os.path.join(data_dir, "train.jsonl")) as f:
            self.samples = [json.loads(l) for l in f]
        self.transform = transforms.Compose([
            transforms.Resize((RESOLUTION, RESOLUTION)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])   # [-1, 1]
        ])
        self.cond_transform = transforms.Compose([
            transforms.Resize((RESOLUTION, RESOLUTION)),
            transforms.ToTensor()                # [0, 1]
        ])

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        photo  = Image.open(os.path.join(self.data_dir, s["image"])).convert("RGB")
        sketch = Image.open(os.path.join(self.data_dir, s["conditioning_image"])).convert("RGB")
        return {
            "pixel_values":      self.transform(photo),
            "conditioning_pixel_values": self.cond_transform(sketch),
            "caption":           s["text"]
        }

dataset    = SketchPhotoDataset(DATA_DIR)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ============ Load Models ============
print("Loading models...")
model_id = "runwayml/stable-diffusion-v1-5"
tokenizer   = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device)
vae          = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device)
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble"
).to(device)

from diffusers import UNet2DConditionModel
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet").to(device)

# ============ Apply LoRA to UNet only ============
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=["to_q", "to_v", "to_k", "to_out.0"],
    lora_dropout=0.1,
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

# Freeze everything except LoRA
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
controlnet.requires_grad_(False)

# ============ Optimizer ============
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=LR
)
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,  # 100
    num_training_steps=MAX_STEPS
)

# ============ Training Loop ============
scaler = torch.cuda.amp.GradScaler() if MIXED_PREC else None
unet.train()
global_step = 0
optimizer.zero_grad()

print(f"Starting training for {MAX_STEPS} steps...")
pbar = tqdm(total=MAX_STEPS, desc="Training")

while global_step < MAX_STEPS:
    for batch in dataloader:
        if global_step >= MAX_STEPS:
            break

        pixel_values = batch["pixel_values"].to(device, dtype=torch.float16 if MIXED_PREC else torch.float32)
        cond_images  = batch["conditioning_pixel_values"].to(device, dtype=torch.float16 if MIXED_PREC else torch.float32)
        captions     = batch["caption"]

        # Tokenize text
        tokens = tokenizer(captions, padding="max_length", max_length=77,
                           truncation=True, return_tensors="pt").input_ids.to(device)

        with torch.cuda.amp.autocast(enabled=MIXED_PREC):
            # Encode image to latent
            latents = vae.encode(pixel_values).latent_dist.sample() * 0.18215

            # Add noise
            noise     = torch.randn_like(latents)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                      (latents.shape[0],), device=device).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Text embedding
            encoder_hidden_states = text_encoder(tokens).last_hidden_state

            # ControlNet forward
            down_samples, mid_sample = controlnet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=encoder_hidden_states,
                controlnet_cond=cond_images,
                return_dict=False
            )

            # UNet forward
            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=encoder_hidden_states,
                down_block_additional_residuals=down_samples,
                mid_block_additional_residual=mid_sample,
            ).sample

            loss = torch.nn.functional.mse_loss(noise_pred.float(), noise.float())
            loss = loss / GRAD_ACCUM

        if MIXED_PREC:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (global_step + 1) % GRAD_ACCUM == 0:
            if MIXED_PREC:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        global_step += 1
        pbar.update(1)
        if global_step % 20 == 0:
            print(f"Step {global_step}/{MAX_STEPS} | Loss: {loss.item() * GRAD_ACCUM:.4f}")

pbar.close()

# ============ Save LoRA weights ============
unet.save_pretrained(OUTPUT_DIR)
print(f"LoRA weights saved to {OUTPUT_DIR}")
'''

In [4]:
with open("train_lora.ipynb", "w") as f:
    f.write(train_script)

print("train_lora.ipynb 寫入完成")

train_lora.ipynb 寫入完成


In [5]:
!python train_lora.ipynb

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading models...
tokenizer_config.json: 100% 806/806 [00:00<00:00, 3.36MB/s]
vocab.json: 100% 1.06M/1.06M [00:00<00:00, 53.2MB/s]
merges.txt: 100% 525k/525k [00:00<00:00, 88.8MB/s]
speci